In [9]:
from __future__ import print_function, division
%matplotlib inline
from matplotlib import pyplot as plt
import json
import random
import numpy as np

import debiaswe as dwe
from   debiaswe.we import WordEmbedding
from   debiaswe.data import load_professions
from   debiaswe.data import load_pairs

# What is hard-debiasing of vector word embeddings and how does it work mathmatically?

## 1. What are word-embeddings?

Word embeddings are how we turn natural language into vectors that computers can work with. There are a number of important word-embedding models, some more complicated than others. From a basic 1:1 model where every word corresponds to one vector, Word-2-Vec and GLoVE embeddings are the most common. These both work by iterating over large corpi of text, and finding context words for every word (that is, words that frequently are seen within proximity to the target word). By doing this for all words, a `m * n`  matrix is created, where `m` is the number of words in the vocabulary and `n` is the size of the embedding vector. 

Word embeddings capture semantic relationships between words, meaning that words with similar meanings are located close to each other in the vector space. This allows for various natural language processing tasks, such as sentiment analysis, machine translation, and information retrieval, to be performed more effectively. By representing words as dense vectors, word embeddings help in reducing the dimensionality of the data and preserving the contextual meaning of words.

Word2Vec is trained using a neural network with a single hidden layer. There are two main architectures for training Word2Vec: Continuous Bag of Words (CBOW) and Skip-gram. 

- **CBOW**: Predicts the target word (center word) from the context words (surrounding words). It maximizes the probability of the target word given the context words.
- **Skip-gram**: Predicts the context words from the target word. It maximizes the probability of the context words given the target word.

The training process involves the following steps:
1. Initialize the weights of the neural network randomly.
2. For each word in the corpus, create training samples based on the chosen architecture (CBOW or Skip-gram).
3. Use the training samples to update the weights of the neural network using backpropagation and gradient descent.
4. After training, the weights of the hidden layer are used as the word embeddings.

The objective function for training Word2Vec is to maximize the log probability of the context words given the target word (or vice versa), which can be computed using softmax.

## 2. What is the problem debiasing tries to solve?

Word embeddings are trained on huge amounts of natural language. For example, Word-2-Vec is commonly trained on all of Google News, or all of Wikipedia. Even more extreme, models like Bidirectional Encoder Representations from Transformers (BERT)
and GPT-models are trained on essentially the entire internet. As you may expect, this training data contains implict and explcit bias and prejudice against certain groups and people. This manifests itself clearly when we look at the problem of "man is to doctor  as woman is to nurse". We do this with vector aritmethic:  $\text{vector}(\text{woman}) + (\text{vector}(\text{doctor}) - \text{vector}(\text{man}))$ gives us nurse. You can see this below where we generate a basic vector for gender, and highlight which professions have a high connotation with gender. This is done by taking the gender vector and using cosine distance to determine which words from the given pairs are closest to it.

In [31]:
# load google news word2vec
E = WordEmbedding('embeddings/w2v_gnews_small.txt')

# load professions
professions = load_professions()
profession_words = [p[0] for p in professions]

gender_vector_simple = E.v('she') - E.v('he') # compute a simple gender vector (just the difference between she and he)

a_gender = E.best_analogies_dist_thresh(gender_vector_simple)

for (a,b,c) in a_gender:
    print(a+"-"+b)

*** Reading data from embeddings/w2v_gnews_small.txt
(26423, 300)
26423 words of dimension 300 : in, for, that, is, ..., Jay, Leroy, Brad, Jermaine
Loaded professions
Format:
word,
definitional female -1.0 -> definitional male 1.0
stereotypical female -1.0 -> stereotypical male 1.0
Computing neighbors
Mean: 10.219732808538016
Median: 7.0
she-he
herself-himself
her-his
woman-man
daughter-son
businesswoman-businessman
girl-boy
actress-actor
chairwoman-chairman
heroine-hero
mother-father
spokeswoman-spokesman
sister-brother
girls-boys
sisters-brothers
queen-king
niece-nephew
councilwoman-councilman
motherhood-fatherhood
women-men
petite-lanky
ovarian_cancer-prostate_cancer
Anne-John
schoolgirl-schoolboy
granddaughter-grandson
aunt-uncle
matriarch-patriarch
twin_sister-twin_brother
mom-dad
lesbian-gay
husband-younger_brother
gal-dude
lady-gentleman
sorority-fraternity
mothers-fathers
grandmother-grandfather
blouse-shirt
soprano-baritone
queens-kings
Jill-Greg
daughters-sons
grandma-grandpa

## 3. What does debiasing do?

## 4. How does debiasing work?

**Method 1: Hard Debaising**

Hard Debiasing (HD) is a method introduced by Bolukbasi et al. (2016) to remove gender bias from word embeddings while preserving meaningful relationships between words. The process involves identifying and neutralizing the difference some between gendered pairs while maintaing the difference that should retain gendered distinctions (e.g. mother/father)

Step-by-step Process:

1. Identify the Gender Subspace Using Singular Value Decomposition

First, select a set of gendered word pairs (e.g. [["woman", "man"], ["girl", "boy"], ["she", "he"], ["mother", "father"]]) - from the debaiswe data, we have definitional_pairs.json. Compute the difference of each of these pairs and stack the difference vectors on top of one another to form a matrix of difference vectors Q. Perform SVD on matrix $Q$ to get $U\sum_{}^{}V^{T}$. The first row of $V^{T}$ corresponds to the top principle component of $Q$, or the direction with the most variance in gender distinction. Consider this the gender variation vector.

2. Neutralize Gender in Non-Gendered Words Using the Top Principle Component 

Comprise a set of words that should be gender neutral (e.g. doctor, nurse, scientist). Use the gender variation vector to remove any variation in gender direction from each word in the set. This is done by projecting a word vector in the set onto the gender variation vector, then subtracting the projection from the vector. This moves the word vector to be orthagonal to the gender variation vector, thus (theoretically) removing its gender variation.

3. Equalize Gendered Pairs

For explicitly gendered words (e.g., he-she, king-queen), compute the mean vector of the word-pair:

$$μ = {w_{male} + w_{female} \over 2}$$

Then adjust each word so they are equidistant from the neutral point, ensuring they remain gender-opposite but symmetrical in the vector space:

$$w'_{male} = μ + {(w_{male} - μ) \over || w_{male} - μ ||}$$

$$w'_{female} = μ - {(w_{female} - μ) \over || w_{female} - μ ||}$$

This ensures that words like he and she, or queen and king, are equidistant from the neutral center but retain their relative positions.

4. Select Words to Debais using Machine Learning

Train a Support Vector Machine using words manually classified as gendered or neutral. Based on the training data, the model predicts which words should be debaised based on their similarity to known gendered words. Then process the selected words in steps 2 and 3.

The Limitations of Hard-Debaising:

While moderately effective for debaising word embeddings, hard debaising has a list of notable drawbacks. First, hard debaising requires manual selection of gendered words to process in step 3. This means that hard debaising cannot run fully automatically. Second, hard debaising only removes bias along a single identified subspace, meaning bias existing in dimensions outside of the identified subspace will be ignored. Finally neutralizing words can effect their semantic relationship to other words. This will alter the meaning of certain words according to their word vectors.

**Method 2: Soft Debaising**
Soft debiasing works by reducing gender bias in word embeddings while preserving as much of the original structure and meaning as possible. Instead of completely removing the gender component (as in hard debiasing), soft debiasing applies a linear transformation to subtly adjust word vectors, balancing the trade-off between bias removal and maintaining semantic relationships.

Step-by-step Process:

1. Identify the Gender Subspace Using Singular Value Decomposition

See step 1 described in the Hard Debaising step-by-step process 

2. Formulate a Linear Transformation

The goal is to find a transformation $T$ that minimizes bias while maintaining the relative distances between words. The desired debiasing transformation $T$ a linear transformation that seeks to preserve pairwise inner products between all the word vectors while minimizing the projection of the gender neutral words onto the gender subspace. This can be formalized as the following optimization problem:

$$min_T = ||(TW)^T(TW) - W^TW||^2_F + \lambda||(TN)^T(TB)||^2_F$$

Where:

* $W$ is the original word embedding matrix
* $N$ is a submatrix of $W$ containing gender-neutral words
* $B$ is the gender subspace found in step 1
* $\lambda$ is the tuning parameter 

For $\lambda$ large, $T$ would remove the projection onto $B$ from all the vectors in $N$, which corresponds exactly to step 2 in method 1. Decreasing $\lambda$ mantains the semantic relationship between word vectors while simultaneously increasing gender bias of gender neutral words.

3. Apply optimized transformation $T$ to the word embedding matrix

The outputter word embeddings will have reduced gender bias but still maintain meaningful relationships between words. Unlike hard debaising, soft debaising does not force complete neutrality, allowing the word embeddings to retain useful gender-related distinctions when necessessary. This comes at the cost of mantaining some gender bias in gender neutral words.





## 5. Example of debiasing

In [32]:

pairs = load_pairs()
pair_words = [p[0] for p in pairs]

difference_vectors = [E.diff(p[0], p[1]) for p in pairs]

D = np.vstack(difference_vectors)

U, S, Vt = np.linalg.svd(D, full_matrices=False)
v_bias = Vt[0]
v_bias /= np.linalg.norm(v_bias)

In [33]:
# profession analysis gender
sp = sorted([(E.v(w).dot(v_bias), w) for w in pair_words])

sp[0:20], sp[-20:]



([(-0.49073303, 'she'),
  (-0.458293, 'her'),
  (-0.41140974, 'herself'),
  (-0.38280421, 'gal'),
  (-0.35907042, 'woman'),
  (-0.32279432, 'girl'),
  (-0.31089544, 'female'),
  (-0.29554015, 'daughter'),
  (-0.2952311, 'mother'),
  (-0.2842507, 'Mary')],
 [(-0.49073303, 'she'),
  (-0.458293, 'her'),
  (-0.41140974, 'herself'),
  (-0.38280421, 'gal'),
  (-0.35907042, 'woman'),
  (-0.32279432, 'girl'),
  (-0.31089544, 'female'),
  (-0.29554015, 'daughter'),
  (-0.2952311, 'mother'),
  (-0.2842507, 'Mary')])